#### Imports

In [ ]:
import kagglehub
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import Sequential, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.metrics import Precision, Recall
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
%pip install ydata-profiling lime
from ydata_profiling import ProfileReport
import lime
import lime.lime_tabular
from sklearn.metrics import accuracy_score


#### File Readings

In [ ]:
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
pd.set_option('display.max_columns', None)


In [ ]:


# Download latest version

print("Path to dataset files:", path)

# Construct the full path to the customers dataset
file_path = os.path.join(path, "olist_orders_dataset.csv")

# Read only the customers dataset into a dataframe
orders_df = pd.read_csv(file_path)
customers_df = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
order_items_df = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
reviews_df = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv"))
products_df = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))

In [ ]:
customers_df = customers_df.drop(columns=['customer_zip_code_prefix', 'customer_city', 'customer_unique_id'])
# products_df = products_df.drop(columns=['product_category_name'])
reviews_df = reviews_df.drop(columns=['review_comment_title', 'review_comment_message'])
# orders_df = orders_df.drop(columns=['order_approved_at', 'order_delivered_carrier_date', 'order_purchase_timestamp'])
# orders_df = orders_df[orders_df['order_status'] == 'delivered']
order_items_df = order_items_df.drop(columns=['seller_id'])

In [ ]:
merged_df = orders_df.merge(order_items_df, on="order_id").merge(products_df, on="product_id").merge(customers_df, on="customer_id").merge(reviews_df, on="order_id")


In [ ]:
merged_df.head()

In [ ]:
merged_df.columns

In [ ]:
merged_df.isna().sum()

In [ ]:
merged_df.columns

In [ ]:
merged_df.iloc[0]

In [ ]:
merged_df.shape

#### Data Cleaning

In [ ]:
merged_df.columns

In [ ]:
merged_df.groupby('order_id').agg(
    total_items_per_order=('order_item_id', 'max'),
    total_price_per_order=('price', 'sum'),
    total_freight_per_order=('freight_value', 'sum')
).reset_index()

In [ ]:
# Convert relevant columns to datetime
merged_df['order_purchase_timestamp'] = pd.to_datetime(merged_df['order_purchase_timestamp'])
merged_df['shipping_limit_date'] = pd.to_datetime(merged_df['shipping_limit_date'])
merged_df['order_approved_at'] = pd.to_datetime(merged_df['order_approved_at'])
merged_df['order_delivered_carrier_date'] = pd.to_datetime(merged_df['order_delivered_carrier_date'])
merged_df['order_delivered_customer_date'] = pd.to_datetime(merged_df['order_delivered_customer_date'])
merged_df['order_estimated_delivery_date'] = pd.to_datetime(merged_df['order_estimated_delivery_date'])
merged_df['review_creation_date'] = pd.to_datetime(merged_df['review_creation_date'])
merged_df['review_answer_timestamp'] = pd.to_datetime(merged_df['review_answer_timestamp'])

# Calculate aggregated features per order
order_grouped = merged_df.groupby('order_id').agg(
    total_items_per_order=('order_item_id', 'max'),
    total_price_per_order=('price', 'sum'),
    total_freight_per_order=('freight_value', 'sum')
).reset_index()

# Merge aggregated features back to the main dataframe
merged_df = merged_df.merge(order_grouped, on='order_id', how='left')



# Create new features
merged_df['purchase_day_of_week'] = merged_df['order_purchase_timestamp'].dt.dayofweek
merged_df['purchase_month'] = merged_df['order_purchase_timestamp'].dt.month
merged_df['purchase_hour'] = merged_df['order_purchase_timestamp'].dt.hour

merged_df['delivery_delay_days'] = (merged_df['order_delivered_customer_date'] - merged_df['order_estimated_delivery_date']).dt.days
merged_df['actual_delivery_time_days'] = (merged_df['order_delivered_customer_date'] - merged_df['order_purchase_timestamp']).dt.days
merged_df['approval_delay_hours'] = (merged_df['order_approved_at'] - merged_df['order_purchase_timestamp']).dt.total_seconds() / 3600 # in hours
merged_df['shipping_delay_hours'] = (merged_df['order_delivered_carrier_date'] - merged_df['order_approved_at']).dt.total_seconds() / 3600 # in hours



merged_df['product_volume_cm3'] = merged_df['product_length_cm'] * merged_df['product_height_cm'] * merged_df['product_width_cm']

# Handle potential division by zero or very small numbers
merged_df['price_per_kg'] = merged_df['price'] / (merged_df['product_weight_g'] / 1000)
merged_df['price_per_cm3'] = merged_df['price'] / merged_df['product_volume_cm3']

# Replace infinite values with NaN
merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)

merged_df['freight_ratio'] = merged_df['freight_value'] / merged_df['price']
# Replace infinite values in freight_ratio as well
merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)


# Calculate average item price per order (after merging total price and items)
merged_df['avg_item_price'] = merged_df['total_price_per_order'] / merged_df['total_items_per_order']
# Correctly assign string labels to sentiment_group
merged_df['sentiment_group'] = merged_df['review_score'].apply(lambda x: 'positive' if x > 3 else ('negative' if x < 3 else 'neutral'))
merged_df['review_response_delay'] = (merged_df['review_answer_timestamp'] - merged_df['review_creation_date']).dt.total_seconds() / 3600 # in hours

merged_df['is_delivered'] = merged_df['order_delivered_customer_date'].notnull()
merged_df['is_shipped'] = merged_df['order_delivered_carrier_date'].notnull()

merged_df['is_carrier_date_missing'] = merged_df['order_delivered_carrier_date'].isna().astype(int)
merged_df['is_delivery_date_missing'] = merged_df['order_delivered_customer_date'].isna().astype(int)

def group_rare_labels(df, col, threshold=0.01):
    """Groups rare labels in a categorical column into an 'Other' category."""
    counts = df[col].value_counts(normalize=True)
    rare_labels = counts[counts < threshold].index
    df[col] = np.where(df[col].isin(rare_labels), 'Other', df[col])
    return df

categorical_columns = ['customer_state', 'product_category_name']
for col in categorical_columns:
    merged_df = group_rare_labels(merged_df, col)

In [ ]:
merged_df.columns

##### Imputation

In [ ]:
for col in ['approval_delay_hours', 'price_per_kg', 'price_per_cm3', 'product_description_lenght', 'product_volume_cm3',
            'product_name_lenght', 'product_category_name', 'product_photos_qty', 'product_weight_g', 'product_length_cm',
            'product_height_cm', 'product_width_cm']:
    if merged_df[col].dtype == 'object':
        merged_df[col] = merged_df[col].fillna('unknown')
    else:
        merged_df[col] = merged_df[col].fillna(merged_df[col].median())

merged_df['order_approved_at'] = merged_df['order_approved_at'].fillna(merged_df['order_purchase_timestamp'])

# Fill null values in 'delivery_delay_days' with 0 for undelivered orders
merged_df['delivery_delay_days'] = merged_df['delivery_delay_days'].fillna(0)
merged_df['actual_delivery_time_days'] = merged_df['actual_delivery_time_days'].fillna(0)
merged_df['shipping_delay_hours'] = merged_df['shipping_delay_hours'].fillna(0)

merged_df['order_delivered_carrier_date'] = merged_df['order_delivered_carrier_date'].fillna(merged_df['order_approved_at'])

# Fill any remaining null values in order_delivered_customer_date with order_estimated_delivery_date
merged_df['order_delivered_customer_date'] = merged_df['order_delivered_customer_date'].fillna(merged_df['order_estimated_delivery_date'])


In [ ]:
merged_df = merged_df.drop(columns=[

    'order_item_id', # Assuming this was also dropped
    # 'review_score', # Dropping review_score as sentiment is engineered from it
    'order_purchase_timestamp'
])

In [ ]:
merged_df.shape

##### Visualization

In [ ]:
ProfileReport(merged_df, title="Profiling Report")

In [ ]:
# Calculate the correlation matrix
correlation_matrix = merged_df.corr(numeric_only=True)

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', annot=False) # Set annot=True for showing values
plt.title('Correlation Matrix of merged_df')
plt.show()

#### Data Engineering Questions

In [ ]:
# 1. Which product category has the highest average review score in each region?

# Group by customer state and product category and calculate the average review score
region_category_reviews = merged_df.groupby(['customer_state', 'product_category_name'])['review_score'].mean().reset_index()

# Find the product category with the highest average review score for each state
highest_reviews_per_region = region_category_reviews.loc[region_category_reviews.groupby('customer_state')['review_score'].idxmax()]

print("Product category with the highest average review score in each region:")
print(highest_reviews_per_region)

# Visualize the results
plt.figure(figsize=(12, 8))
sns.barplot(x='customer_state', y='review_score', hue='product_category_name', data=highest_reviews_per_region)
plt.title('Product Category with Highest Average Review Score per State')
plt.xlabel('Customer State')
plt.ylabel('Average Review Score')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Product Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.ylim(3.5, 5)
plt.show()

In [ ]:
# 2. What is the relationship between delivery delay and review sentiment?

# Visualize the relationship between delivery delay (continuous) and review score (categorical)
plt.figure(figsize=(12, 8))
sns.boxplot(x='review_score', y='delivery_delay_days', data=merged_df)
plt.title('Relationship between Delivery Delay and Review Score')
plt.xlabel('Review Score')
plt.ylabel('Delivery Delay (Days)')
plt.ylim(-50, 50) # Set the y-axis limits
plt.show()

In [ ]:
# 3. Which regions show the highest proportion of negative reviews?

# Group by customer state and sentiment group
state_sentiment = merged_df.groupby(['customer_state', 'sentiment_group']).size().unstack(fill_value=0)

# Calculate the proportion of negative reviews
state_sentiment['negative_proportion'] = state_sentiment['negative'] / state_sentiment.sum(axis=1)

# Sort by the proportion of negative reviews
highest_negative_reviews = state_sentiment.sort_values(by='negative_proportion', ascending=False).reset_index()

print("Regions with the highest proportion of negative reviews:")
print(highest_negative_reviews[['customer_state', 'negative_proportion']])

# Visualize the results
plt.figure(figsize=(12, 8))
sns.barplot(x='customer_state', y='negative_proportion', data=highest_negative_reviews)
plt.title('Proportion of Negative Reviews per State')
plt.xlabel('Customer State')
plt.ylabel('Proportion of Negative Reviews')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#### Predictive Modeling Task

In [ ]:
# Drop unnecessary IDs that don't add predictive power
X = merged_df.drop(columns=[
    'sentiment_group',
    'order_id',
    'customer_id',
    'product_id',
    'review_score',
    'review_id'])

# Convert datetime columns into useful numeric features
date_cols = [
    'shipping_limit_date',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'review_creation_date',
    'review_answer_timestamp'
]

for col in date_cols:
    X[f'{col}_weekday'] = X[col].dt.weekday
    X[f'{col}_hour'] = X[col].dt.hour
    X[f'{col}_month'] = X[col].dt.month
    X.drop(columns=[col], inplace=True)

# Split features
categorical_features = X.select_dtypes(include='object').columns.tolist()
numerical_features = X.select_dtypes(include=np.number).columns.tolist()


# Preprocessing
preprocessor = ColumnTransformer([
    ('num', 'passthrough', numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_processed = preprocessor.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, merged_df['sentiment_group'], test_size=0.2, random_state=42 # Use the correct 'sentiment_group' column as target
)

In [ ]:
X_train.shape

In [ ]:

# Initialize and train different models
models = {
    'Random Forest': RandomForestClassifier()
}

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train) # Convert sparse matrix to dense array for models that require it
    trained_models[name] = model
    print(f"{name} training complete.")

Now that the models are trained, I will evaluate their performance on the test data.

In [ ]:
# Evaluate models
results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test) # Convert sparse matrix to dense array for models that require it
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    results[name] = {'accuracy': accuracy, 'report': report}
    print(f"Evaluation for {name}:\nAccuracy: {accuracy}\nClassification Report:\n{report}\n{'-'*50}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions from the best model
best_model_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = trained_models[best_model_name]
y_pred_best_model = best_model.predict(X_test) # X_test is already processed, no need for .toarray()

# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred_best_model)

# Display the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

Based on the evaluation results, we can now predict sentiment using the trained models.

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'shipping_limit_date', 'price',
       'freight_value', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'customer_state', 'review_id', 'review_score', 'review_creation_date',
       'review_answer_timestamp']

#### Model Explainability

In [ ]:
i=1
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=preprocessor.get_feature_names_out(),
    class_names=['negative', 'neutral', 'positive'],
    discretize_continuous=True
)

exp = explainer_lime.explain_instance(
    X_test[i],
    best_model.predict_proba,
    num_features=20,
    labels=[0, 1, 2] # Request explanations for all class indices
)
exp.show_in_notebook()

exp.as_pyplot_figure(label=0)  # Plot explanation for class index 0 (negative)
exp.as_pyplot_figure(label=1)  # Plot explanation for class index 1 (neutral)
exp.as_pyplot_figure(label=2)  # Plot explanation for class index 2 (positive)

#### Model Test

In [ ]:
def clean_data(X_test_fin):
  # merged_df = pd.DataFrame(X_test_fin, columns=preprocessor.get_feature_names_out())
  merged_df = X_test_fin
  # Convert relevant columns to datetime
  merged_df['order_purchase_timestamp'] = pd.to_datetime(merged_df['order_purchase_timestamp'])
  merged_df['shipping_limit_date'] = pd.to_datetime(merged_df['shipping_limit_date'])
  merged_df['order_approved_at'] = pd.to_datetime(merged_df['order_approved_at'])
  merged_df['order_delivered_carrier_date'] = pd.to_datetime(merged_df['order_delivered_carrier_date'])
  merged_df['order_delivered_customer_date'] = pd.to_datetime(merged_df['order_delivered_customer_date'])
  merged_df['order_estimated_delivery_date'] = pd.to_datetime(merged_df['order_estimated_delivery_date'])
  merged_df['review_creation_date'] = pd.to_datetime(merged_df['review_creation_date'])
  merged_df['review_answer_timestamp'] = pd.to_datetime(merged_df['review_answer_timestamp'])

  # Calculate aggregated features per order
  order_grouped = merged_df.groupby('order_id').agg(
      total_items_per_order=('order_item_id', 'max'),
      total_price_per_order=('price', 'sum'),
      total_freight_per_order=('freight_value', 'sum')
  ).reset_index()

  # Merge aggregated features back to the main dataframe
  merged_df = merged_df.merge(order_grouped, on='order_id', how='left')



  # Create new features
  merged_df['purchase_day_of_week'] = merged_df['order_purchase_timestamp'].dt.dayofweek
  merged_df['purchase_month'] = merged_df['order_purchase_timestamp'].dt.month
  merged_df['purchase_hour'] = merged_df['order_purchase_timestamp'].dt.hour

  merged_df['delivery_delay_days'] = (merged_df['order_delivered_customer_date'] - merged_df['order_estimated_delivery_date']).dt.days
  merged_df['actual_delivery_time_days'] = (merged_df['order_delivered_customer_date'] - merged_df['order_purchase_timestamp']).dt.days
  merged_df['approval_delay_hours'] = (merged_df['order_approved_at'] - merged_df['order_purchase_timestamp']).dt.total_seconds() / 3600 # in hours
  merged_df['shipping_delay_hours'] = (merged_df['order_delivered_carrier_date'] - merged_df['order_approved_at']).dt.total_seconds() / 3600 # in hours



  merged_df['product_volume_cm3'] = merged_df['product_length_cm'] * merged_df['product_height_cm'] * merged_df['product_width_cm']

  # Handle potential division by zero or very small numbers
  merged_df['price_per_kg'] = merged_df['price'] / (merged_df['product_weight_g'] / 1000)
  merged_df['price_per_cm3'] = merged_df['price'] / merged_df['product_volume_cm3']

  # Replace infinite values with NaN
  merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)

  merged_df['freight_ratio'] = merged_df['freight_value'] / merged_df['price']
  # Replace infinite values in freight_ratio as well
  merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)


  # Calculate average item price per order (after merging total price and items)
  merged_df['avg_item_price'] = merged_df['total_price_per_order'] / merged_df['total_items_per_order']
  # Correctly assign string labels to sentiment_group
  merged_df['sentiment_group'] = merged_df['review_score'].apply(lambda x: 'positive' if x > 3 else ('negative' if x < 3 else 'neutral'))
  merged_df['review_response_delay'] = (merged_df['review_answer_timestamp'] - merged_df['review_creation_date']).dt.total_seconds() / 3600 # in hours

  merged_df['is_delivered'] = merged_df['order_delivered_customer_date'].notnull()
  merged_df['is_shipped'] = merged_df['order_delivered_carrier_date'].notnull()

  merged_df['is_carrier_date_missing'] = merged_df['order_delivered_carrier_date'].isna().astype(int)
  merged_df['is_delivery_date_missing'] = merged_df['order_delivered_customer_date'].isna().astype(int)

  def group_rare_labels(df, col, threshold=0.01):
      """Groups rare labels in a categorical column into an 'Other' category."""
      counts = df[col].value_counts(normalize=True)
      rare_labels = counts[counts < threshold].index
      df[col] = np.where(df[col].isin(rare_labels), 'Other', df[col])
      return df

  categorical_columns = ['customer_state', 'product_category_name']
  for col in categorical_columns:
      merged_df = group_rare_labels(merged_df, col)

  for col in ['approval_delay_hours', 'price_per_kg', 'price_per_cm3', 'product_description_lenght', 'product_volume_cm3',
            'product_name_lenght', 'product_category_name', 'product_photos_qty', 'product_weight_g', 'product_length_cm',
            'product_height_cm', 'product_width_cm']:
    if merged_df[col].dtype == 'object':
        merged_df[col] = merged_df[col].fillna('unknown')
    else:
        merged_df[col] = merged_df[col].fillna(merged_df[col].median())

  merged_df['order_approved_at'] = merged_df['order_approved_at'].fillna(merged_df['order_purchase_timestamp'])

  # Fill null values in 'delivery_delay_days' with 0 for undelivered orders
  merged_df['delivery_delay_days'] = merged_df['delivery_delay_days'].fillna(0)
  merged_df['actual_delivery_time_days'] = merged_df['actual_delivery_time_days'].fillna(0)
  merged_df['shipping_delay_hours'] = merged_df['shipping_delay_hours'].fillna(0)

  merged_df['order_delivered_carrier_date'] = merged_df['order_delivered_carrier_date'].fillna(merged_df['order_approved_at'])

  # Fill any remaining null values in order_delivered_customer_date with order_estimated_delivery_date
  merged_df['order_delivered_customer_date'] = merged_df['order_delivered_customer_date'].fillna(merged_df['order_estimated_delivery_date'])
  merged_df = merged_df.drop(columns=[

    'order_item_id', # Assuming this was also dropped
    'order_purchase_timestamp'
  ])

  X = merged_df.drop(columns=[
    'sentiment_group',
    'order_id',
    'customer_id',
    'product_id',
    'review_score',
    'review_id'])

  # Convert datetime columns into useful numeric features
  date_cols = [
      'shipping_limit_date',
      'order_approved_at',
      'order_delivered_carrier_date',
      'order_delivered_customer_date',
      'order_estimated_delivery_date',
      'review_creation_date',
      'review_answer_timestamp'
  ]

  for col in date_cols:
      X[f'{col}_weekday'] = X[col].dt.weekday
      X[f'{col}_hour'] = X[col].dt.hour
      X[f'{col}_month'] = X[col].dt.month
      X.drop(columns=[col], inplace=True)

  return X

In [ ]:

def evaluate_and_print_accuracy(model, X_test, y_test):
    """Evaluates the model and prints predictions with correctness."""
    # Ensure y_test is array-like
    if isinstance(y_test, str):
        y_test = [y_test]
    elif not isinstance(y_test, (list, np.ndarray, pd.Series)):
        y_test = list(y_test)
    flag = False
    y_pred = model.predict(X_test)

    # Handle the case when y_pred is a single value
    if np.isscalar(y_pred):
        y_pred = [y_pred]

    accuracy = accuracy_score(y_test, y_pred)
    if accuracy == 1:
        flag = True
    print(f"\nOverall Accuracy: {accuracy:.2f}\n")
    print("Detailed results:")
    print("-" * 40)

    for actual, predicted in zip(y_test, y_pred):
        result = "✅ Right" if actual == predicted else "❌ Wrong"
        print(f"Actual: {actual:<10} | Predicted: {predicted:<10} | {result}")
    return flag


In [ ]:
# xtest = {
#     'order_id': 'e481f51cbdc54678b7cc49136f2d6af7',
#     'customer_id': '9ef432eb6251297304e76186b10a928d',
#     'order_status': 'delivered',
#     'order_purchase_timestamp': pd.Timestamp('2017-10-02 11:07:15'),
#     'order_approved_at': pd.Timestamp('2017-10-04 19:55:00'),
#     'order_delivered_carrier_date': pd.Timestamp('2017-10-10 21:25:13'),
#     'order_delivered_customer_date': pd.Timestamp('2017-10-18 00:00:00'),
#     'order_estimated_delivery_date': pd.Timestamp('2017-10-25 00:00:00'),
#     'order_item_id': np.int64(1),
#     'product_id': '87285b34884572647811a353c7ac498a',
#     'shipping_limit_date': pd.Timestamp('2017-10-06 11:07:15'),
#     'price': np.float64(29.99),
#     'freight_value': np.float64(8.72),
#     'product_category_name': 'utilidades_domesticas',
#     'product_name_lenght': np.float64(40.0),
#     'product_description_lenght': np.float64(268.0),
#     'product_photos_qty': np.float64(4.0),
#     'product_weight_g': np.float64(500.0),
#     'product_length_cm': np.float64(19.0),
#     'product_height_cm': np.float64(8.0),
#     'product_width_cm': np.float64(13.0),
#     'customer_state': 'SP',
#     'review_id': 'a54f0611adc9ed256b57ede6b6eb5114',
#     'review_score': np.int64(4),
#     'review_creation_date': pd.Timestamp('2017-10-11 00:00:00'),
#     'review_answer_timestamp': pd.Timestamp('2017-10-12 03:43:48')
# }


In [ ]:
test_df = pd.read_csv('random_orders_sample.csv')
actual_df = pd.read_csv('actual_results.csv')
sum, sum2 = 0, 0
for index, row in test_df.iterrows():
  row_df = pd.DataFrame([row])

  xtest_df = clean_data(row_df)
  xtest_processed = preprocessor.transform(xtest_df)
  ytest = actual_df.loc[index, 'sentiment_group']

  if evaluate_and_print_accuracy(best_model, xtest_processed, ytest):
    sum += 1
  sum2 += 1



In [ ]:
print(f"Accuracy: {sum/sum2}")